# Data Analysis with Streamlit, Polars and Plotly

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinayaseshu/data_analysis_with_streamlit/blob/main/run_app.ipynb)

This notebook:
1. Clones the repo and installs dependencies.
2. Launches the Streamlit app (`app.py`) and gives you a link to open it.
3. Runs the same Polars + Plotly analysis directly in the notebook.

Run the cells from top to bottom.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/vinayaseshu/data_analysis_with_streamlit.git"
REPO_DIR = "data_analysis_with_streamlit"

if not os.path.exists("app.py"):
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Working directory:", os.getcwd())

## 2. Launch the Streamlit app

Starts Streamlit in the background on port 8501.
- **Google Colab:** prints a link served through Colab's own port proxy.
- **Elsewhere:** opens a temporary public URL with a [Cloudflare quick tunnel](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/). If you're running Jupyter locally you can also just open http://localhost:8501.

Re-running this cell restarts the app.

In [ ]:
import time, urllib.request, platform, re, stat

PORT = 8501
subprocess.run(["pkill", "-f", "streamlit run app.py"], check=False)
log = open("streamlit.log", "w")
app_proc = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py",
     "--server.port", str(PORT), "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    stdout=log, stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        urllib.request.urlopen(f"http://localhost:{PORT}/_stcore/health", timeout=1)
        print(f"Streamlit is running on port {PORT}")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Streamlit did not start; see streamlit.log")

try:
    from google.colab.output import eval_js
    url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
    print("Open the app:", url)
except ImportError:
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    binary = f"./cloudflared-linux-{arch}"
    if not os.path.exists(binary):
        urllib.request.urlretrieve(
            f"https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-{arch}", binary)
        os.chmod(binary, os.stat(binary).st_mode | stat.S_IEXEC)
    tunnel_log = open("cloudflared.log", "w")
    tunnel_proc = subprocess.Popen([binary, "tunnel", "--url", f"http://localhost:{PORT}"],
                                   stdout=tunnel_log, stderr=subprocess.STDOUT)
    url = None
    for _ in range(60):
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            url = m.group(0)
            break
        time.sleep(1)
    print("Open the app:", url or f"http://localhost:{PORT} (tunnel failed, see cloudflared.log)")

## 3. Analysis in the notebook

The same sample dataset and analysis as the app, using Polars and Plotly. To use your own data, set `DATA_PATH` to a CSV or Parquet file.

In [ ]:
import numpy as np
import polars as pl
import plotly.express as px

DATA_PATH = None  # e.g. "my_data.csv"

def sample_data(n: int = 1000) -> pl.DataFrame:
    rng = np.random.default_rng(42)
    dates = pl.date_range(pl.date(2024, 1, 1), pl.date(2024, 1, 1) + pl.duration(days=n - 1), eager=True)
    return pl.DataFrame({
        "date": dates,
        "region": rng.choice(["North", "South", "East", "West"], n),
        "product": rng.choice(["A", "B", "C"], n),
        "units": rng.integers(1, 100, n),
        "price": rng.normal(50, 12, n).round(2),
    }).with_columns(revenue=(pl.col("units") * pl.col("price")).round(2))

if DATA_PATH is None:
    df = sample_data()
elif DATA_PATH.endswith(".parquet"):
    df = pl.read_parquet(DATA_PATH)
else:
    df = pl.read_csv(DATA_PATH, try_parse_dates=True, infer_schema_length=10000)

numeric_cols = [c for c, t in df.schema.items() if t.is_numeric()]
categorical_cols = [c for c, t in df.schema.items() if t in (pl.String, pl.Categorical, pl.Boolean)]
print(df.shape)
df.head()

### Summary statistics

In [ ]:
df.describe()

### Correlation matrix

In [ ]:
corr = df.select(numeric_cols).drop_nulls().corr()
px.imshow(corr.to_numpy(), x=numeric_cols, y=numeric_cols, text_auto=".2f",
          color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Correlation matrix")

### Distribution

In [ ]:
px.histogram(df.to_pandas(), x=numeric_cols[-1], color=categorical_cols[0] if categorical_cols else None,
             nbins=40, marginal="box", title=f"Distribution of {numeric_cols[-1]}")

### Group by

In [ ]:
if categorical_cols:
    key, val = categorical_cols[0], numeric_cols[-1]
    grouped = df.group_by(key).agg(
        pl.col(val).sum().alias(f"{val}_sum"),
        pl.col(val).mean().round(2).alias(f"{val}_mean"),
        pl.len().alias("count"),
    ).sort(key)
    display(grouped)
    display(px.bar(grouped.to_pandas(), x=key, y=f"{val}_sum", title=f"Total {val} by {key}"))

### Time series

In [ ]:
temporal_cols = [c for c, t in df.schema.items() if t.is_temporal()]
if temporal_cols:
    t, val = temporal_cols[0], numeric_cols[-1]
    monthly = df.group_by_dynamic(t, every="1mo").agg(pl.col(val).sum()).sort(t)
    display(px.line(monthly.to_pandas(), x=t, y=val, markers=True, title=f"Monthly {val}"))

## 4. Stop the app

In [ ]:
subprocess.run(["pkill", "-f", "streamlit run app.py"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
print("Stopped")